# 01_EDA_Exploratorio

EDA exploratorio de tweets recolectados sobre precios de alimentos en Lima.

**Entrada principal:**
- `bronce_tweets_precios_lima.json`

**Entrada opcional:**
- `tweets_precios_lima_final.csv`, si ya contiene la columna `sentimiento`.

Este notebook genera análisis de:
- volumen de tweets,
- duplicados y nulos,
- fechas,
- distritos,
- sentimiento,
- likes y retweets,
- términos de búsqueda,
- palabras frecuentes.

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

RUTA_JSON = Path("bronce_tweets_precios_lima.json")
RUTA_CSV_FINAL = Path("tweets_precios_lima_final.csv")

print("Carpeta actual:", Path.cwd())
print("Existe JSON:", RUTA_JSON.exists())
print("Existe CSV final:", RUTA_CSV_FINAL.exists())

## 1. Carga de datos desde JSON

In [ ]:
if not RUTA_JSON.exists():
    raise FileNotFoundError(
        f"No se encontró {RUTA_JSON}. "
        "Coloca este notebook en la misma carpeta donde está tu JSON."
    )

with open(RUTA_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Filas iniciales:", len(df))
print("Columnas:", df.columns.tolist())
df.head()

## 2. Limpieza básica y enriquecimiento

In [ ]:
# Normalizar columnas esperadas
if "comentario" not in df.columns:
    raise ValueError("No existe la columna 'comentario' en el JSON.")

df["comentario"] = df["comentario"].astype(str)

if "fecha" in df.columns:
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["anio"] = df["fecha"].dt.year
    df["mes"] = df["fecha"].dt.month
    df["periodo"] = df["fecha"].dt.to_period("M").astype(str)
    df["dia_semana"] = df["fecha"].dt.day_name()
else:
    print("Advertencia: no existe columna fecha.")

# Duplicados
filas_antes = len(df)

if "id" in df.columns:
    df = df.drop_duplicates(subset=["id"]).copy()

df = df.drop_duplicates(subset=["comentario"]).reset_index(drop=True)

print("Filas antes:", filas_antes)
print("Filas después de quitar duplicados:", len(df))
print("Duplicados eliminados:", filas_antes - len(df))

# Si el JSON no trae sentimiento pero existe el CSV final, lo traemos desde ahí por comentario
if "sentimiento" not in df.columns and RUTA_CSV_FINAL.exists():
    df_sent = pd.read_csv(RUTA_CSV_FINAL)
    if {"comentario", "sentimiento"}.issubset(df_sent.columns):
        df_sent = df_sent[["comentario", "sentimiento"]].drop_duplicates(subset=["comentario"])
        df = df.merge(df_sent, on="comentario", how="left")
        print("Se agregó sentimiento desde tweets_precios_lima_final.csv")
    else:
        print("El CSV final existe, pero no tiene comentario/sentimiento.")

# Texto auxiliar
df["longitud_texto"] = df["comentario"].str.len()
df["num_palabras"] = df["comentario"].str.split().str.len()

df.head()

## 3. Resumen general

In [ ]:
resumen = pd.DataFrame({
    "indicador": [
        "tweets_total",
        "fecha_min",
        "fecha_max",
        "distritos_distintos",
        "comentarios_unicos",
        "tweets_con_distrito",
        "tweets_sin_distrito",
        "likes_total",
        "retweets_total"
    ],
    "valor": [
        len(df),
        df["fecha"].min() if "fecha" in df.columns else None,
        df["fecha"].max() if "fecha" in df.columns else None,
        df["distrito"].nunique(dropna=True) if "distrito" in df.columns else None,
        df["comentario"].nunique(),
        df["distrito"].notna().sum() if "distrito" in df.columns else None,
        df["distrito"].isna().sum() if "distrito" in df.columns else None,
        df["likes"].sum() if "likes" in df.columns else None,
        df["retweets"].sum() if "retweets" in df.columns else None,
    ]
})

resumen

In [ ]:
# Nulos por columna
nulos = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "columna", 0: "nulos"})
)
nulos["porcentaje_nulos"] = nulos["nulos"] / len(df) * 100
nulos.sort_values("nulos", ascending=False)

## 4. Distribución del sentimiento

In [ ]:
if "sentimiento" in df.columns:
    conteo_sent = df["sentimiento"].value_counts(dropna=False).rename_axis("sentimiento").reset_index(name="cantidad")
    conteo_sent["porcentaje"] = conteo_sent["cantidad"] / conteo_sent["cantidad"].sum() * 100
    display(conteo_sent)

    plt.figure(figsize=(6, 4))
    conteo_sent_plot = df["sentimiento"].map({0: "Negativo", 1: "Positivo"}).fillna("Sin clasificar").value_counts()
    conteo_sent_plot.plot(kind="bar")
    plt.title("Distribución del sentimiento")
    plt.xlabel("Sentimiento")
    plt.ylabel("Cantidad de tweets")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No existe columna sentimiento. Corre primero el script de clasificación o asegúrate de tener tweets_precios_lima_final.csv.")

## 5. Evolución mensual de tweets

In [ ]:
if "periodo" in df.columns:
    mensual = (
        df.groupby("periodo")
        .size()
        .reset_index(name="tweets")
        .sort_values("periodo")
    )

    display(mensual.head())
    display(mensual.tail())

    plt.figure(figsize=(12, 5))
    plt.plot(mensual["periodo"], mensual["tweets"], marker="o")
    plt.title("Evolución mensual del volumen de tweets")
    plt.xlabel("Periodo")
    plt.ylabel("Cantidad de tweets")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
else:
    print("No se puede graficar por periodo porque no existe fecha.")

In [ ]:
if {"periodo", "sentimiento"}.issubset(df.columns):
    mensual_sent = (
        df.groupby(["periodo", "sentimiento"])
        .size()
        .reset_index(name="tweets")
        .pivot(index="periodo", columns="sentimiento", values="tweets")
        .fillna(0)
        .rename(columns={0: "negativo", 1: "positivo"})
        .reset_index()
    )

    display(mensual_sent.head())

    plt.figure(figsize=(12, 5))
    if "negativo" in mensual_sent.columns:
        plt.plot(mensual_sent["periodo"], mensual_sent["negativo"], marker="o", label="Negativo")
    if "positivo" in mensual_sent.columns:
        plt.plot(mensual_sent["periodo"], mensual_sent["positivo"], marker="o", label="Positivo")
    plt.title("Evolución mensual por sentimiento")
    plt.xlabel("Periodo")
    plt.ylabel("Cantidad de tweets")
    plt.xticks(rotation=90)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No hay columnas suficientes para evolución mensual por sentimiento.")

## 6. Análisis por distrito

In [ ]:
if "distrito" in df.columns:
    top_distritos = (
        df["distrito"]
        .fillna("Sin distrito detectado")
        .value_counts()
        .head(20)
        .reset_index()
    )
    top_distritos.columns = ["distrito", "tweets"]
    display(top_distritos)

    plt.figure(figsize=(10, 6))
    top_distritos.sort_values("tweets").plot(x="distrito", y="tweets", kind="barh", legend=False)
    plt.title("Top 20 distritos por cantidad de tweets")
    plt.xlabel("Cantidad de tweets")
    plt.ylabel("Distrito")
    plt.tight_layout()
    plt.show()
else:
    print("No existe columna distrito.")

In [ ]:
if {"distrito", "sentimiento"}.issubset(df.columns):
    distrito_sent = (
        df.dropna(subset=["distrito"])
        .groupby("distrito")
        .agg(
            tweets=("comentario", "count"),
            positivos=("sentimiento", lambda x: (x == 1).sum()),
            negativos=("sentimiento", lambda x: (x == 0).sum()),
            sentimiento_promedio=("sentimiento", "mean")
        )
        .reset_index()
        .sort_values("tweets", ascending=False)
    )

    distrito_sent["pct_positivo"] = distrito_sent["positivos"] / distrito_sent["tweets"] * 100
    distrito_sent["pct_negativo"] = distrito_sent["negativos"] / distrito_sent["tweets"] * 100

    display(distrito_sent.head(20))

    # Graficar solo distritos con mínimo 10 tweets para evitar ruido
    distrito_plot = distrito_sent[distrito_sent["tweets"] >= 10].sort_values("pct_negativo", ascending=False).head(15)

    if len(distrito_plot) > 0:
        plt.figure(figsize=(10, 6))
        distrito_plot.sort_values("pct_negativo").plot(
            x="distrito",
            y="pct_negativo",
            kind="barh",
            legend=False
        )
        plt.title("Top distritos con mayor % de sentimiento negativo")
        plt.xlabel("% negativo")
        plt.ylabel("Distrito")
        plt.tight_layout()
        plt.show()
    else:
        print("No hay suficientes distritos con mínimo 10 tweets.")
else:
    print("No hay columnas suficientes para analizar sentimiento por distrito.")

## 7. Términos de búsqueda

In [ ]:
if "termino_busqueda" in df.columns:
    terminos = (
        df["termino_busqueda"]
        .value_counts()
        .head(25)
        .reset_index()
    )
    terminos.columns = ["termino_busqueda", "tweets"]
    display(terminos)

    plt.figure(figsize=(10, 7))
    terminos.sort_values("tweets").plot(x="termino_busqueda", y="tweets", kind="barh", legend=False)
    plt.title("Top 25 términos de búsqueda con más tweets recolectados")
    plt.xlabel("Cantidad de tweets")
    plt.ylabel("Término")
    plt.tight_layout()
    plt.show()
else:
    print("No existe columna termino_busqueda.")

## 8. Likes, retweets y longitud de texto

In [ ]:
metricas = [c for c in ["likes", "retweets", "longitud_texto", "num_palabras"] if c in df.columns]

for col in metricas:
    print(f"\nResumen de {col}:")
    display(df[col].describe())

    plt.figure(figsize=(7, 4))
    df[col].dropna().clip(upper=df[col].quantile(0.99)).hist(bins=30)
    plt.title(f"Distribución de {col} - recortado al p99")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()

In [ ]:
if {"likes", "retweets"}.issubset(df.columns):
    top_interaccion = df.copy()
    top_interaccion["interacciones"] = top_interaccion["likes"].fillna(0) + top_interaccion["retweets"].fillna(0)

    cols_mostrar = [c for c in ["fecha", "comentario", "distrito", "sentimiento", "likes", "retweets", "interacciones"] if c in top_interaccion.columns]
    display(top_interaccion.sort_values("interacciones", ascending=False)[cols_mostrar].head(15))
else:
    print("No existen likes y retweets para analizar interacción.")

## 9. Palabras frecuentes en comentarios

In [ ]:
STOPWORDS_ES = {
    "a", "al", "algo", "ante", "antes", "como", "con", "contra", "cual", "cuando",
    "de", "del", "desde", "donde", "durante", "e", "el", "ella", "ellas", "ellos",
    "en", "entre", "era", "eres", "es", "esa", "esas", "ese", "eso", "esos",
    "esta", "está", "estaba", "estan", "están", "este", "esto", "estos", "fue",
    "ha", "han", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas",
    "más", "me", "mi", "mis", "muy", "no", "nos", "o", "para", "pero", "por",
    "porque", "que", "qué", "se", "si", "sí", "sin", "sobre", "su", "sus", "te",
    "tiene", "tienen", "todo", "todos", "tu", "un", "una", "unas", "uno", "unos",
    "y", "ya", "yo", "rt", "https", "http", "co", "x", "lima", "peru", "perú"
}

def quitar_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFD", str(texto))
        if unicodedata.category(c) != "Mn"
    )

def limpiar_texto(texto):
    texto = quitar_tildes(str(texto).lower())
    texto = re.sub(r"http\S+|www\S+", " ", texto)
    texto = re.sub(r"@\w+", " ", texto)
    texto = re.sub(r"#", " ", texto)
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

textos_limpios = df["comentario"].apply(limpiar_texto)

palabras = []
for texto in textos_limpios:
    for palabra in texto.split():
        if len(palabra) >= 3 and palabra not in STOPWORDS_ES:
            palabras.append(palabra)

freq_palabras = pd.Series(palabras).value_counts().head(30).reset_index()
freq_palabras.columns = ["palabra", "frecuencia"]

display(freq_palabras)

plt.figure(figsize=(10, 7))
freq_palabras.sort_values("frecuencia").plot(x="palabra", y="frecuencia", kind="barh", legend=False)
plt.title("Top 30 palabras frecuentes")
plt.xlabel("Frecuencia")
plt.ylabel("Palabra")
plt.tight_layout()
plt.show()

In [ ]:
if "sentimiento" in df.columns:
    for valor, nombre in [(0, "negativo"), (1, "positivo")]:
        textos_filtrados = df.loc[df["sentimiento"] == valor, "comentario"].apply(limpiar_texto)

        palabras_filtradas = []
        for texto in textos_filtrados:
            for palabra in texto.split():
                if len(palabra) >= 3 and palabra not in STOPWORDS_ES:
                    palabras_filtradas.append(palabra)

        freq = pd.Series(palabras_filtradas).value_counts().head(20).reset_index()
        freq.columns = ["palabra", "frecuencia"]

        print(f"\nTop palabras en tweets con sentimiento {nombre}:")
        display(freq)

        if len(freq) > 0:
            plt.figure(figsize=(9, 6))
            freq.sort_values("frecuencia").plot(x="palabra", y="frecuencia", kind="barh", legend=False)
            plt.title(f"Top 20 palabras en sentimiento {nombre}")
            plt.xlabel("Frecuencia")
            plt.ylabel("Palabra")
            plt.tight_layout()
            plt.show()
else:
    print("No existe sentimiento para separar palabras por clase.")

## 10. Exportables del EDA

In [ ]:
CARPETA_SALIDA = Path("salidas_eda")
CARPETA_SALIDA.mkdir(exist_ok=True)

# Dataset limpio
df.to_csv(CARPETA_SALIDA / "tweets_limpios_eda.csv", index=False, encoding="utf-8-sig")

# Resumen mensual
if "periodo" in df.columns:
    mensual.to_csv(CARPETA_SALIDA / "resumen_mensual_tweets.csv", index=False, encoding="utf-8-sig")

# Resumen por distrito
if "distrito" in df.columns:
    top_distritos.to_csv(CARPETA_SALIDA / "top_distritos_tweets.csv", index=False, encoding="utf-8-sig")

if {"distrito", "sentimiento"}.issubset(df.columns):
    distrito_sent.to_csv(CARPETA_SALIDA / "resumen_distrito_sentimiento.csv", index=False, encoding="utf-8-sig")

if "termino_busqueda" in df.columns:
    terminos.to_csv(CARPETA_SALIDA / "top_terminos_busqueda.csv", index=False, encoding="utf-8-sig")

freq_palabras.to_csv(CARPETA_SALIDA / "top_palabras_frecuentes.csv", index=False, encoding="utf-8-sig")

print("Archivos exportados en:", CARPETA_SALIDA.resolve())

## 11. Conclusiones rápidas

Completa esta sección después de ejecutar el notebook.

Ideas para redactar:
- Cantidad final de tweets analizados.
- Periodos con mayor volumen de conversación.
- Distritos más mencionados.
- Proporción de sentimiento negativo y positivo.
- Términos o productos asociados a mayor conversación.
- Palabras frecuentes que explican la preocupación por precios.